# 00 — Setup & Authentication

This notebook verifies your environment is ready to talk to the FortyGuard tOS Enterprise API.

**Before running:** copy `.env.example` to `.env` at the repo root and paste your API key into it.

We will:
1. Load the API key from `.env`
2. Instantiate the Python client
3. Make one lightweight call (credit usage) to confirm auth works

In [3]:
import sys
from pathlib import Path

# Find the repository root.
repo_root = Path.cwd().parent

# Make the local FortyGuard package importable.
sys.path.insert(0, str(repo_root))

print("Python:", sys.executable)
print("Repository:", repo_root)
print("fortyguard folder exists:", (repo_root / "fortyguard").exists())

Python: c:\Users\user\temperature-api-quickstart\venv\Scripts\python.exe
Repository: c:\Users\user\temperature-api-quickstart
fortyguard folder exists: True


In [4]:
from dotenv import load_dotenv
import os

env_path = repo_root / ".env"

print(".env exists:", env_path.exists())

load_dotenv(env_path)

print("API key loaded:", bool(os.getenv("FORTYGUARD_API_KEY")))
print("Base URL:", os.getenv("FORTYGUARD_BASE_URL"))

.env exists: True
API key loaded: True
Base URL: https://api.fortyguard.com


In [5]:
from fortyguard import FortyGuardClient

print("FortyGuardClient imported successfully!")

FortyGuardClient imported successfully!


In [6]:
client = FortyGuardClient()

print("Client created successfully!")
print("Base URL:", client.base_url)
print("API key:", client.api_key[:6] + "…" if client.api_key else "(missing)")

Client created successfully!
Base URL: https://api.fortyguard.com
API key: accd46…


In [7]:
from datetime import date, timedelta

end_date = date.today().isoformat()
start_date = (date.today() - timedelta(days=30)).isoformat()

print(f"Checking usage from {start_date} to {end_date}...")

usage = client.fetch_api_key_custom_usage(
    start_date=start_date,
    end_date=end_date
)

print("\n✅ FortyGuard API responded successfully!")

date_range = usage.get("date_range", {})

print(
    "Window:",
    date_range.get("date_range_formatted")
    or f"{start_date} → {end_date}"
)

print("Credits used:", usage.get("total_credits_used"))

print("\nActivity breakdown:")
for row in usage.get("activity_breakdown", []):
    print(
        f"  {row.get('name'):>28}: "
        f"{row.get('credits')} credits "
        f"over {row.get('count')} calls"
    )

Checking usage from 2026-07-31 to 2026-08-30...

✅ FortyGuard API responded successfully!
Window: Jul 31, 2026 – Aug 30, 2026
Credits used: 0

Activity breakdown:


In [ ]:
# Hit the credits endpoint as a cheap auth check.
from datetime import date, timedelta

end_date   = date.today().isoformat()
start_date = (date.today() - timedelta(days=30)).isoformat()

usage = client.fetch_api_key_custom_usage(start_date=start_date, end_date=end_date)

date_range = usage.get('date_range', {})
print(f"Window      : {date_range.get('date_range_formatted') or f'{start_date} → {end_date}'}")
print(f"Credits used: {usage.get('total_credits_used')}")

for row in usage.get('activity_breakdown', []):
    print(f"  {row.get('name'):>28}: {row.get('credits')} credits over {row.get('count')} calls")

In [ ]:
print("=== HEATMAP SUMMARY ===")

# Fetch map data from the API
map_data = client.fetch_temperature_data()  # or the appropriate method call

print("=== HEATMAP SUMMARY ===")

# Fetch map data from the API
map_data = client.fetch_temperature_data()  # or the appropriate method call

print("Number of tiles:", len(map_data["features"]))

temps = [
    f["properties"]["average_temperature"]
    for f in map_data["features"]
    if f["properties"].get("average_temperature") is not None
]

print("Minimum:", min(temps))
print("Maximum:", max(temps))
print("Mean:", sum(temps) / len(temps))

print("Minimum:", min(temps))
print("Maximum:", max(temps))
print("Mean:", sum(temps) / len(temps))

If the cell above printed your plan and remaining credits, you're ready. Continue with `01_create_heatmap.ipynb`.